# Project: Audience Segmentation for E-commerce

#### Data Cleaning & Exploratory Data Analysis

**Goal:** clean the raw transaction-level Online Retail dataset and shape it into a reliable, one-row-per-transaction table that can be aggregated into an **RFM (Recency, Frequency, Monetary)** table for customer segmentation in the next notebook.

**Workflow:**
1. Setup
2. Load & inspect raw data
3. Fix data types
4. Handle missing values
5. Remove duplicate records
6. Feature engineering & country-level view
7. Handle returns, cancellations & invalid transactions
8. Build final RFM-ready dataset
9. Key insights summary & export

## 1. Setup

In [6]:
import pandas as pd
from datetime import timedelta
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load & Inspect Raw Data

In [10]:
# Read the raw transaction-level data
data = pd.read_csv("data/online_retail.csv")
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/online_retail.csv'

In [ ]:
# Sanity-check the tail of the file too (different country, later date)
data.tail()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,"12,680.00",France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,"12,680.00",France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,"12,680.00",France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,"12,680.00",France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,"12,680.00",France


In [ ]:
# Check shape and data types before doing anything else
print(data.shape)
data.dtypes

(541909, 8)


InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

**Observations**
- `541,909` rows, `8` columns — one row per product line within an invoice, not one row per order.
- `InvoiceDate` is stored as `object` (text), and `CustomerID` as `float64` even though it's an identifier — both need to be fixed before any aggregation.
- `InvoiceNo` starting with `"C"` is the UCI dataset's convention for a **cancelled/returned order** — we'll need to handle this explicitly for RFM (a cancellation shouldn't count as a purchase).

## 3. Fix Data Types

In [ ]:
# CustomerID is an identifier, not a number to do math on -> convert to nullable integer, then string
data["CustomerID"] = data["CustomerID"].astype("Int64").astype("string")

# InvoiceDate needs to be an actual datetime so we can compute Recency later
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

data.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID     string[python]
Country                object
dtype: object

## 4. Handle Missing Values

In [ ]:
# Check missing values across all columns
data.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

**Observations**
- `CustomerID` is missing for `135,080` rows (**~24.9%** of the data). These transactions can't be tied to any individual, so they are unusable for a customer-level RFM analysis and must be dropped.
- `Description` is missing for `1,454` rows — this is a product-label field that doesn't affect RFM (which only needs `CustomerID`, `InvoiceDate`, `InvoiceNo`, `Quantity`, `UnitPrice`), so instead of dropping these rows we just fill the label so it doesn't break downstream text operations.

In [ ]:
# Drop rows with no CustomerID - these can never be attributed to a customer for RFM
data.dropna(subset=["CustomerID"], inplace=True)

# Number of unique customers remaining
data["CustomerID"].nunique()

4372

In [ ]:
# Fill missing product descriptions with an explicit label (not "0", which reads as a numeric/product code)
data["Description"] = data["Description"].fillna("Unknown")

## 5. Remove Duplicate Records

In [ ]:
# Check for fully duplicated rows (same invoice, product, quantity, price, customer, timestamp)
data.duplicated().sum()

np.int64(5225)

**Observation** — `5,225` rows are *exact* duplicates (identical on every column, including the timestamp to the minute). This isn't "the customer bought the same item twice" — that would still differ in row identity in this dataset structure — it's almost certainly a **double-scan / duplicate-entry artifact** at checkout or in data export. Left in place, these would silently inflate a customer's `Frequency` and `Monetary` values in the RFM table, so we remove them here rather than just reporting the count.

In [ ]:
# Drop the duplicates
data.drop_duplicates(inplace=True)
data.reset_index(drop=True, inplace=True)

print(f"Rows after removing duplicates: {data.shape[0]:,}")

Rows after removing duplicates: 401,604


## 6. Feature Engineering — TotalAmount & Country-Level View

Create two derived columns:
- **TotalAmount** = Quantity x UnitPrice — the line-level revenue figure that Monetary (and country-level revenue) will be built from.
- **OrderDate** = `InvoiceDate` truncated to the date (no time component).

In [ ]:
data["TotalAmount"] = data["Quantity"] * data["UnitPrice"]
data["OrderDate"] = data["InvoiceDate"].dt.date
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,OrderDate
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12-01
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12-01
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01


In [ ]:
# Revenue, customers and orders by country
total_country = (
    data.groupby("Country")
        .agg(
            UniqueCustomers=("CustomerID", "nunique"),
            TotalQuantity=("Quantity", "sum"),
            TotalOrders=("InvoiceNo", "nunique"),
            AvgUnitPrice=("UnitPrice", "mean"),
            TotalAmount=("TotalAmount", "sum")
        )
        .sort_values("TotalAmount", ascending=False)
        .round(2)
        .reset_index()
)
total_country.head(15)

,Country,UniqueCustomers,TotalQuantity,TotalOrders,AvgUnitPrice,TotalAmount
0,United Kingdom,3950,3994870,19857,3.27,"6,747,156.15"
1,Netherlands,9,200128,101,2.74,"284,661.54"
2,EIRE,3,136187,319,5.11,"250,001.78"
3,Germany,95,117341,603,3.97,"221,509.47"
4,France,87,109806,458,5.05,"196,626.05"
5,Australia,9,83643,69,3.22,"137,009.77"
6,Switzerland,21,29778,71,3.50,"55,739.40"
7,Spain,31,26817,105,4.99,"54,756.03"
8,Belgium,25,23152,119,3.64,"40,910.96"
9,Sweden,8,35632,46,3.91,"36,585.41"


**Observation** — the dataset is heavily skewed toward the **United Kingdom**, which accounts for the vast majority of customers, orders and revenue. Any downstream segmentation should either (a) treat the UK as the primary market and other countries as a long tail, or (b) explicitly note that RFM thresholds derived from the full dataset will mostly reflect UK purchasing behavior.

## 7. Handle Returns, Cancellations & Invalid Transactions

In [ ]:
# Check for negative TotalAmount values
(data["TotalAmount"] < 0).any()

np.True_

In [ ]:
# Inspect the most extreme negative transactions
negative_amounts = data[data["TotalAmount"] < 0]
negative_amounts.sort_values("TotalAmount").head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,OrderDate
401132,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446,United Kingdom,"-168,469.60",2011-12-09
37516,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346,United Kingdom,"-77,183.60",2011-01-18
157406,C556445,M,Manual,-1,2011-06-10 15:31:00,"38,970.00",15098,United Kingdom,"-38,970.00",2011-06-10
312797,C573079,M,Manual,-2,2011-10-27 14:15:00,"4,161.06",12536,France,"-8,322.12",2011-10-27
119809,C551685,POST,POSTAGE,-1,2011-05-03 12:51:00,"8,142.75",16029,United Kingdom,"-8,142.75",2011-05-03
119916,C551699,M,Manual,-1,2011-05-03 14:12:00,"6,930.00",16029,United Kingdom,"-6,930.00",2011-05-03
110949,C550456,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,-3114,2011-04-18 13:08:00,2.10,15749,United Kingdom,"-6,539.40",2011-04-18
110947,C550456,85123A,WHITE HANGING HEART T-LIGHT HOLDER,-1930,2011-04-18 13:08:00,2.55,15749,United Kingdom,"-4,921.50",2011-04-18
110945,C550456,48185,DOORMAT FAIRY CAKE,-670,2011-04-18 13:08:00,6.75,15749,United Kingdom,"-4,522.50",2011-04-18
189323,C560372,M,Manual,-1,2011-07-18 12:26:00,"4,287.63",17448,United Kingdom,"-4,287.63",2011-07-18


In [ ]:
data.loc[data["TotalAmount"] < 0, "TotalAmount"].describe().round(2)

count      8,872.00
mean         -68.61
std        2,022.87
min     -168,469.60
25%          -17.00
50%           -8.50
75%           -3.30
max           -0.12
Name: TotalAmount, dtype: float64

In [ ]:
# Share of all line items that are negative-value
negative_pct = (data["TotalAmount"] < 0).mean() * 100
print(f"Negative transactions: {negative_pct:.2f}%")

Negative transactions: 2.21%


In [ ]:
# Negative revenue by country
negative_by_country = (
    data[data["TotalAmount"] < 0]
    .groupby("Country")
    .agg(
        NegativeTransactions=("TotalAmount", "count"),
        NegativeAmount=("TotalAmount", "sum")
    )
    .sort_values("NegativeAmount")
)
negative_by_country.head(10)

,NegativeTransactions,NegativeAmount
Country,,
United Kingdom,7501,"-537,868.49"
EIRE,247,"-15,260.68"
France,148,"-12,308.26"
Singapore,7,"-12,158.90"
Germany,453,"-7,168.93"
Spain,48,"-6,802.53"
Portugal,18,"-4,380.08"
Japan,37,"-2,075.75"
USA,112,"-1,849.47"


In [ ]:
# Flag cancelled orders - by UCI convention InvoiceNo starting with "C" is a cancellation
data["IsCancellation"] = data["InvoiceNo"].astype(str).str.startswith("C")
data["IsCancellation"].value_counts()

IsCancellation
False    392732
True       8872
Name: count, dtype: int64

In [ ]:
cancellations = data[data["IsCancellation"]].copy()
cancellations[["InvoiceNo", "CustomerID", "StockCode", "Quantity", "UnitPrice", "TotalAmount"]].head()

,InvoiceNo,CustomerID,StockCode,Quantity,UnitPrice,TotalAmount
141,C536379,14527,D,-1,27.50,-27.50
154,C536383,15311,35004C,-1,4.65,-4.65
235,C536391,17548,22556,-12,1.65,-19.80
236,C536391,17548,21984,-24,0.29,-6.96
237,C536391,17548,21983,-24,0.29,-6.96


In [ ]:
# Cancellation activity per customer 
cancellation_summary = (
    data[data["IsCancellation"]]
    .groupby("CustomerID")
    .agg(
        CancellationCount=("InvoiceNo", "nunique"),
        CancelledQuantity=("Quantity", "sum"),
        CancelledAmount=("TotalAmount", "sum")
    )
    .reset_index()
)
cancellation_summary.head()

,CustomerID,CancellationCount,CancelledQuantity,CancelledAmount
0,12346,1,-74215,"-77,183.60"
1,12352,3,-66,-960.63
2,12359,2,-10,-127.05
3,12362,3,-17,-71.65
4,12365,1,-1,-320.69


**Observation** — every negative `TotalAmount` row corresponds to a cancelled invoice (the negative-amount count matches the cancellation count exactly), so cancellations are the *only* source of negative revenue here — there's no separate data-entry issue to chase. 

In [ ]:
# Also check for zero-value line items that are NOT cancellations (e.g. free samples, price-entry errors,
# "POST"/"M"/"D" adjustment codes with UnitPrice = 0). These carry no monetary signal and would
# artificially dilute a customer's average spend if left in.
zero_value = data[(data["TotalAmount"] == 0) & (~data["IsCancellation"])]
print(f"Zero-value, non-cancellation rows: {len(zero_value):,} "
      f"({len(zero_value) / len(data):.2%} of remaining data)")
zero_value.head()

Zero-value, non-cancellation rows: 40 (0.01% of remaining data)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,OrderDate,IsCancellation
6842,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.00,12647,Germany,0.00,2010-12-05,False
22619,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.00,16560,United Kingdom,0.00,2010-12-16,False
25551,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,0.00,14911,EIRE,0.00,2010-12-21,False
29374,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.00,13081,United Kingdom,0.00,2011-01-06,False
29376,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.00,13081,United Kingdom,0.00,2011-01-06,False


These zero-value rows add no Monetary signal but *do* represent a real interaction (an invoice line), so we drop them only from the monetary calculation context, not by silently deleting invoices — in practice, since `TotalAmount = 0` never changes a sum, we simply need to make sure they aren't miscounted as cancellations; no further action is required here.

### Visualization

In [ ]:
# Top 10 countries by revenue
top10 = total_country.head(10)

fig = px.bar(
    top10,
    x="TotalAmount",
    y="Country",
    orientation="h",
    title="Top 10 Countries by Revenue",
    labels={"TotalAmount": "Total Revenue", "Country": ""}
)
fig.update_traces(marker_color="#BDE0FE")
fig.update_layout(template="simple_white", yaxis=dict(categoryorder="total ascending"))
fig.show()

In [ ]:
top10_quant = total_country.sort_values("TotalQuantity", ascending=False).head(10)

fig = px.bar(
    top10_quant, 
    x="TotalQuantity",
    y="Country",
    title="Top 10 Countries by Quantity",
    labels={"TotalQuantity": "Total Quantity", "Country": ""},
)


fig.update_traces(marker_color="#B5DBB4")
fig.update_layout(template="simple_white", yaxis=dict(categoryorder="total ascending"), coloraxis_showscale=False)
fig.show()

In [ ]:
total_country["RevenuePerCustomer"] = (total_country["TotalAmount"] / total_country["UniqueCustomers"]).round(2)
top10_rpc = total_country.sort_values("RevenuePerCustomer", ascending=False).head(10)

fig = px.bar(
    top10_rpc,
    x="RevenuePerCustomer",
    y="Country",
    orientation="h",
    title="Top 10 Countries by Revenue per Customer",
    labels={"RevenuePerCustomer": "Revenue per Customer", "Country": ""},
)
fig.update_traces(marker_color="#CDB4DB")
fig.update_layout(template="simple_white", yaxis=dict(categoryorder="total ascending"), coloraxis_showscale=False)
fig.show()

**Observation** — while the UK leads on total revenue and customer count, several smaller markets (e.g. Netherlands, EIRE, Australia) show a much higher **revenue per customer**, driven by a small number of large wholesale-style buyers rather than broad customer adoption.

In [ ]:
# How many countries are represented overall
print(f"{data['Country'].nunique()} unique countries")
data["Country"].unique()

37 unique countries


array(['United Kingdom', 'France', 'Australia', 'Netherlands', 'Germany',
       'Norway', 'EIRE', 'Switzerland', 'Spain', 'Poland', 'Portugal',
       'Italy', 'Belgium', 'Lithuania', 'Japan', 'Iceland',
       'Channel Islands', 'Denmark', 'Cyprus', 'Sweden', 'Austria',
       'Israel', 'Finland', 'Greece', 'Singapore', 'Lebanon',
       'United Arab Emirates', 'Saudi Arabia', 'Czech Republic', 'Canada',
       'Unspecified', 'Brazil', 'USA', 'European Community', 'Bahrain',
       'Malta', 'RSA'], dtype=object)

In [ ]:
# Distribution overview for the numeric fields feeding RFM (Quantity, UnitPrice, TotalAmount)
data.describe().round(2).T

,count,mean,min,25%,50%,75%,max,std
Quantity,"401,604.00",12.18,"-80,995.00",2.00,5.00,12.00,"80,995.00",250.28
InvoiceDate,401604,2011-07-10 12:08:23.848567552,2010-12-01 08:26:00,2011-04-06 15:02:00,2011-07-29 15:40:00,2011-10-20 11:58:30,2011-12-09 12:50:00,NaN
UnitPrice,"401,604.00",3.47,0.00,1.25,1.95,3.75,"38,970.00",69.76
TotalAmount,"401,604.00",20.61,"-168,469.60",4.25,11.70,19.80,"168,469.60",430.35


**Observation** — `Quantity` and `TotalAmount` have very heavy tails (max in the tens of thousands), driven by a small number of bulk/wholesale orders. 

## 8. Final Clean Dataset for RFM

**Cancellations are kept, not dropped.**

In [ ]:
# Quick check of the purchase / cancellation split before export
print(f"Total rows: {len(data):,}")
print(f"Purchase rows: {(~data['IsCancellation']).sum():,}")
print(f"Cancellation rows: {data['IsCancellation'].sum():,}")
print(f"Unique customers: {data['CustomerID'].nunique():,}")

Total rows: 401,604
Purchase rows: 392,732
Cancellation rows: 8,872
Unique customers: 4,372


Compute the **reference date** (the day after the most recent *purchase* — cancellations don't count as activity for this purpose). This is the anchor point Recency will be measured against in the RFM notebook: `Recency = reference_date - last purchase date`.

In [ ]:
reference_date = data.loc[~data["IsCancellation"], "InvoiceDate"].max() + timedelta(days=1)
print(f"Reference date: {reference_date}")

Reference date: 2011-12-10 12:50:00


In [ ]:
# Final sanity checks before handing this off to the RFM notebook
assert data["CustomerID"].isnull().sum() == 0
assert data.duplicated().sum() == 0
assert data.loc[data["IsCancellation"], "TotalAmount"].le(0).all(), "cancellations should never be positive"
assert data.loc[~data["IsCancellation"], "TotalAmount"].ge(0).all(), "purchases should never be negative"

data[["InvoiceNo", "CustomerID", "InvoiceDate", "OrderDate", "Quantity", "UnitPrice", "TotalAmount", "IsCancellation", "Country"]].head()

,InvoiceNo,CustomerID,InvoiceDate,OrderDate,Quantity,UnitPrice,TotalAmount,IsCancellation,Country
0,536365,17850,2010-12-01 08:26:00,2010-12-01,6,2.55,15.30,False,United Kingdom
1,536365,17850,2010-12-01 08:26:00,2010-12-01,6,3.39,20.34,False,United Kingdom
2,536365,17850,2010-12-01 08:26:00,2010-12-01,8,2.75,22.00,False,United Kingdom
3,536365,17850,2010-12-01 08:26:00,2010-12-01,6,3.39,20.34,False,United Kingdom
4,536365,17850,2010-12-01 08:26:00,2010-12-01,6,3.39,20.34,False,United Kingdom


In [ ]:
data["OrderDate"].min

NameError: name 'data' is not defined

## 9. Key Insights Summary

- **Scope**: raw file has `541,909` line items; after removing rows with no `CustomerID` (`~24.9%` of rows — these can't be attributed to a customer), exact duplicates (`5,225` rows), the data is now clean at the transaction-line level.
- **Cancellations are kept, not dropped**: `8,905` line items (`2.19%`) are cancellations, flagged via `IsCancellation` rather than removed. They account for *all* negative revenue in the dataset (no unrelated data-entry errors were mixed in). Keeping them lets the RFM notebook net purchases against returns for Monetary, while Frequency and Recency are still computed from purchase rows only.
- **Duplicates**: `5,225` exact duplicate rows were found and removed — left in, they would have inflated Frequency and Monetary for the affected customers.
- **Geography**: the United Kingdom dominates on revenue, orders and customer count; a few smaller countries have high revenue-per-customer driven by large wholesale-style orders rather than broad adoption — worth segmenting UK vs. rest-of-world if the business asks for it later.
- **Outliers**: `Quantity` and `TotalAmount` are heavy-tailed thanks to bulk orders. 
- **Output**: the full cleaned dataset — purchases *and* cancellations, one row per line item, with valid `CustomerID`, `InvoiceDate`, `OrderDate`, `Quantity`, `UnitPrice`, `TotalAmount`, `IsCancellation` and `ReferenceDate` — no duplicates. `OrderDate` (date only, no time) is included specifically to make date filtering in Power BI simple. `ReferenceDate` travels with the export so the RFM notebook anchors Recency to the exact same cutoff used here, instead of recalculating it from a possibly-different snapshot.

In [ ]:
# Attach the reference date to every row so the RFM notebook doesn't need to recompute it
# (keeps Recency calculations consistent with the exact cleaned dataset used here)
data["ReferenceDate"] = reference_date

# Export the full cleaned dataset - purchases AND cancellations - for the RFM notebook.
# The RFM notebook uses IsCancellation to decide what feeds Frequency/Recency vs. Monetary.
data.to_csv("data/online_retail_clean.csv", index=False)
print("Saved: data/online_retail_clean.csv")
print(f"Reference date: {reference_date}")

Saved: data/online_retail_clean.csv
Reference date: 2011-12-10 12:50:00
